# Sanity check

Run this first. It verifies the installation, reports which backends are
available, and confirms the two agree bit-for-bit.

If anything here fails, nothing else in `notebooks/` will be meaningful.

In [ ]:
import sys
from pathlib import Path

# Run from anywhere: notebooks/ is a sibling of src/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

In [ ]:
import platform
import sys

print("python  ", sys.version.split()[0], platform.machine())
print("platform", platform.system(), platform.release())

import tfidf_stability
print("package ", tfidf_stability.__version__)

from tfidf_stability._native import native_available
print("native  ", "available" if native_available() else "absent (reference-only install)")

## The floating-point environment

The reference backend is normative, but a build with fast-math enabled would
silently reassociate arithmetic and break every reproducibility claim. The guard
is checked at run time as well as at compile time.

In [ ]:
from tfidf_stability.persistence.manifest import environment_block

for key, value in sorted(environment_block().items()):
    print(f"{key:22} {value}")

## The pipeline digest

`scripts/snapshot.py` computes one SHA-256 over every number the pipeline
produces on a fixed corpus. CI compares this string across Linux, macOS and
Windows, at three optimisation levels, under both backends, and requires them
all to be identical.

If your value differs from a colleague's, a published number has moved.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, str(REPO / "scripts" / "snapshot.py"), "--verbose"],
    capture_output=True, text=True, cwd=REPO,
)
print(result.stdout or result.stderr)

## Reference vs native, bit-for-bit

Equality is asserted on **raw bit patterns**, never with a tolerance. A tolerance
would admit the divergence this study is about.

The comparison requires a compiled extension. A reference-only install is
supported, and is reported rather than asserted against: CI installs the
notebook job with `TFIDF_BUILD_NATIVE=OFF`, so it does not exercise this section.


In [ ]:
import struct

from tfidf_stability._native import native_available, unavailable_reason
from tfidf_stability.preprocessing.pipeline import PreprocessingPipeline
from tfidf_stability.similarity.cosine import cosine_against_corpus
from tfidf_stability.utils.io import read_jsonl
from tfidf_stability.utils.numerics import Reduction, same_bits
from tfidf_stability.vectorisation.tfidf import TfidfVectoriser

records = list(read_jsonl(REPO / "tests" / "fixtures" / "mini_corpus.jsonl"))
pipeline = PreprocessingPipeline()
features = [pipeline.preprocess(r["text"]) for r in records]
model = TfidfVectoriser().fit(features, [r["doc_id"] for r in records])
documents = [model.document(i) for i in range(model.n_documents)]

query = TfidfVectoriser.transform_query(["quick", "brown", "fox"], model)
reference = cosine_against_corpus(query, documents, model.norms, Reduction.NAIVE)

print(f"{model.n_documents} documents, |V| = {model.n_features}")
for doc_id, score in sorted(zip(model.doc_ids, reference), key=lambda p: -p[1])[:5]:
    print(f"  {doc_id:6} {score!r:24} {struct.pack('<d', score).hex()}")

if not native_available():
    print()
    print("native backend absent, nothing to compare against:")
    print(f"  {unavailable_reason()}")
    print("Build the extension (`pip install -e .`) to exercise this section.")
else:
    import numpy as np

    from tfidf_stability._native import _tfidf_native as nat

    index = nat.NativeIndex(
        np.array(model.matrix.indptr, dtype=np.int64),
        np.array(model.matrix.indices, dtype=np.int32),
        np.array(model.matrix.values, dtype=np.float64),
        model.n_documents,
        model.n_features,
        int(nat.REDUCTION[Reduction.NAIVE.value]),
    )
    native = index.score(
        np.array(query.indices, dtype=np.int32),
        np.array(query.values, dtype=np.float64),
        int(nat.ALGORITHM["taat"]),
    )

    compared = 0
    for i, (a, b) in enumerate(zip(reference, native, strict=True)):
        assert same_bits(a, b), (
            f"document {i} ({model.doc_ids[i]}): reference {a!r} "
            f"({struct.pack('<d', a).hex()}) != native {b!r} "
            f"({struct.pack('<d', b).hex()})"
        )
        compared += 1

    assert compared == model.n_documents, "every document must be compared"
    print()
    print(f"reference and native agree on all {compared} scores, bit for bit")
